# Assignment 1: Log Mel Filter Banks & Speech Classification

> **Google Colab**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Install dependencies (safe to run locally too)
!pip install torchaudio thop -q

In [ ]:
import os
import time
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F_nn
from torchaudio import functional as F_audio
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from typing import Optional

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Part 1: LogMelFilterBanks

In [ ]:
class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.center = center
        self.return_complex = return_complex
        self.onesided = onesided
        self.normalize_stft = normalize_stft
        self.pad_mode = pad_mode
        self.power = power
        self.f_min_hz = f_min_hz
        self.f_max_hz = f_max_hz if f_max_hz is not None else samplerate / 2.0
        self.norm_mel = norm_mel
        self.mel_scale = mel_scale

        # Register as buffers so they move to GPU automatically with .to(device)
        self.register_buffer('window', torch.hann_window(self.window_length))
        self.register_buffer('mel_fbanks', self._init_melscale_fbanks())

    def _init_melscale_fbanks(self):
        return F_audio.melscale_fbanks(
            n_freqs=self.n_fft // 2 + 1,
            f_min=self.f_min_hz,
            f_max=self.f_max_hz,
            n_mels=self.n_mels,
            sample_rate=self.samplerate,
            norm=self.norm_mel,
            mel_scale=self.mel_scale,
        )

    def spectrogram(self, x):
        return torch.stft(
            x,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.window_length,
            window=self.window,
            center=self.center,
            pad_mode=self.pad_mode,
            normalized=self.normalize_stft,
            onesided=self.onesided,
            return_complex=self.return_complex,
        )

    def forward(self, x):
        """
        Args:
            x: (batch, time) audio signal
        Returns:
            (batch, n_mels, n_frames) log mel filterbanks
        """
        spec = self.spectrogram(x)                  # (batch, n_freqs, n_frames) complex
        power_spec = torch.abs(spec) ** self.power  # (batch, n_freqs, n_frames)
        # mel_fbanks: (n_freqs, n_mels) -> .T: (n_mels, n_freqs)
        mel_spec = torch.matmul(self.mel_fbanks.T, power_spec)  # (batch, n_mels, n_frames)
        return torch.log(mel_spec + 1e-6)

### Verification: compare with `torchaudio.transforms.MelSpectrogram`

In [ ]:
# Synthetic test signal: 440 Hz + 880 Hz sine waves, 1 second at 16 kHz
sr = 16000
t = torch.linspace(0, 1, sr)
signal = (torch.sin(2 * np.pi * 440 * t) + 0.5 * torch.sin(2 * np.pi * 880 * t)).unsqueeze(0)  # (1, 16000)

# Reference: torchaudio MelSpectrogram
melspec_transform = torchaudio.transforms.MelSpectrogram(hop_length=160, n_mels=80)
melspec = melspec_transform(signal)  # (1, 80, n_frames)
ref = torch.log(melspec + 1e-6)

# Our implementation
logmel_transform = LogMelFilterBanks()
logmelbanks = logmel_transform(signal)  # (1, 80, n_frames)

print(f'Shapes match : {ref.shape == logmelbanks.shape}  {ref.shape}')
print(f'Values close : {torch.allclose(ref, logmelbanks, atol=1e-5)}')
print(f'Max abs diff : {(ref - logmelbanks).abs().max().item():.2e}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(logmelbanks[0].detach().numpy(), aspect='auto', origin='lower')
axes[0].set_title('LogMelFilterBanks (our implementation)')
axes[0].set_xlabel('Frame'); axes[0].set_ylabel('Mel bin')
axes[1].imshow(ref[0].detach().numpy(), aspect='auto', origin='lower')
axes[1].set_title('log(MelSpectrogram + 1e-6)  (torchaudio reference)')
axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Mel bin')
plt.tight_layout()
plt.savefig('part1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Dataset — YES / NO binary classification

In [ ]:
# Dataset will be downloaded here (~2.3 GB). Excluded from git via .gitignore.
DATA_ROOT = './data'

class YesNoDataset(SPEECHCOMMANDS):
    LABELS = {'no': 0, 'yes': 1}
    TARGET_LENGTH = 16000  # 1 second at 16 kHz

    def __init__(self, root=DATA_ROOT, subset='training', download=True):
        super().__init__(root, download=download, subset=subset)
        self._walker = [
            w for w in self._walker
            if os.path.basename(os.path.dirname(w)) in self.LABELS
        ]

    def __getitem__(self, idx):
        waveform, sample_rate, label, *_ = super().__getitem__(idx)
        waveform = self._pad_or_trim(waveform)
        return waveform.squeeze(0), self.LABELS[label]  # (time,), int

    @staticmethod
    def _pad_or_trim(waveform):
        length = waveform.shape[-1]
        target = YesNoDataset.TARGET_LENGTH
        if length > target:
            return waveform[..., :target]
        if length < target:
            return F_nn.pad(waveform, (0, target - length))
        return waveform


print('Loading dataset (downloading if needed)...')
train_dataset = YesNoDataset(subset='training')
val_dataset   = YesNoDataset(subset='validation')
test_dataset  = YesNoDataset(subset='testing')
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2  # set to 0 if you get multiprocessing errors on Windows

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))

x_batch, y_batch = next(iter(train_loader))
print(f'Waveform batch : {x_batch.shape}  (batch, time)')
print(f'Label batch    : {y_batch.shape}  values: {y_batch.unique().tolist()}')

## Part 3: Model

In [ ]:
class SpeechCNN(nn.Module):
    """
    Simple Conv1d classifier for binary yes/no speech recognition.
    Feature extraction (LogMelFilterBanks) is part of the model,
    so raw waveforms are passed as input.

    Architecture: LogMelFilterBanks -> 3x Conv1d -> AdaptiveAvgPool -> Linear
    """
    def __init__(self, n_mels: int = 80, groups: int = 1):
        super().__init__()
        self.feature_extractor = LogMelFilterBanks(n_mels=n_mels)
        self.backbone = nn.Sequential(
            nn.Conv1d(n_mels, 64,  kernel_size=3, padding=1, groups=groups),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64,    64,  kernel_size=3, padding=1, groups=groups),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64,    128, kernel_size=3, padding=1, groups=groups),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        # x: (batch, time)
        features = self.feature_extractor(x)  # (batch, n_mels, n_frames)
        out = self.backbone(features)          # (batch, 128, 1)
        out = out.squeeze(-1)                  # (batch, 128)
        return self.classifier(out)            # (batch, 2)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_backbone_parameters(self):
        trainable = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
        return trainable(self.backbone) + trainable(self.classifier)


# Sanity check
model = SpeechCNN(n_mels=80, groups=1).to(device)
dummy = torch.zeros(2, 16000).to(device)
out = model(dummy)
print(f'Output shape     : {out.shape}  (batch, 2 classes)')
print(f'Total params     : {model.count_parameters():,}')
print(f'Backbone params  : {model.count_backbone_parameters():,}')

In [ ]:
# FLOPs calculation (backbone only, since torch.stft is not profiled by thop)
try:
    from thop import profile as thop_profile
    n_frames = 101  # approx frames for 1s waveform at hop_length=160
    dummy_mel = torch.zeros(1, 80, n_frames)
    flops, params = thop_profile(model.backbone, inputs=(dummy_mel.to(device),), verbose=False)
    print(f'Backbone FLOPs  : {flops / 1e6:.2f} M')
    print(f'Backbone params : {params / 1e3:.2f} K')
except Exception as e:
    print(f'thop error: {e}')
    print(f'Backbone params (manual): {model.count_backbone_parameters():,}')

## Part 4: Training pipeline

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    start = time.time()
    for waveform, labels in loader:
        waveform, labels = waveform.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(waveform), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * waveform.size(0)
    return total_loss / len(loader.dataset), time.time() - start


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for waveform, labels in loader:
        waveform, labels = waveform.to(device), labels.to(device)
        preds = model(waveform).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

In [ ]:
def run_experiment(n_mels=80, groups=1, epochs=10, lr=1e-3):
    """
    Trains SpeechCNN and returns (model, history_dict, test_accuracy).
    history_dict keys: 'train_loss', 'val_acc', 'epoch_time'
    """
    model = SpeechCNN(n_mels=n_mels, groups=groups).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_acc': [], 'epoch_time': []}

    print(f'n_mels={n_mels}, groups={groups} | params={model.count_parameters():,}')
    for epoch in range(1, epochs + 1):
        train_loss, epoch_time = train_epoch(model, train_loader, optimizer)
        val_acc = evaluate(model, val_loader)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(epoch_time)
        print(f'  Epoch {epoch:2d}/{epochs}  loss={train_loss:.4f}  val_acc={val_acc:.4f}  time={epoch_time:.1f}s')

    test_acc = evaluate(model, test_loader)
    print(f'  => Test accuracy: {test_acc:.4f}')
    return model, history, test_acc

In [ ]:
EPOCHS = 10

print('=== Baseline: n_mels=80, groups=1 ===')
baseline_model, baseline_history, baseline_test_acc = run_experiment(n_mels=80, groups=1, epochs=EPOCHS)

## Part 5: Experiment — varying `n_mels`

Fix `groups=1`, vary `n_mels` ∈ {20, 40, 80}.

In [ ]:
N_MELS_LIST = [20, 40, 80]
n_mels_results = {}

for n_mels in N_MELS_LIST:
    print(f'\n=== n_mels={n_mels} ===')
    _, history, test_acc = run_experiment(n_mels=n_mels, groups=1, epochs=EPOCHS)
    n_mels_results[n_mels] = {'history': history, 'test_acc': test_acc}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Effect of n_mels (groups=1)', fontsize=14)

for n_mels, result in n_mels_results.items():
    h = result['history']
    axes[0].plot(h['train_loss'],  label=f'n_mels={n_mels}')
    axes[1].plot(h['val_acc'],     label=f'n_mels={n_mels}')
    axes[2].plot(h['epoch_time'],  label=f'n_mels={n_mels}')

for ax, title, ylabel in zip(axes,
    ['Train Loss', 'Val Accuracy', 'Epoch Time (s)'],
    ['Loss', 'Accuracy', 'Seconds']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.legend()

plt.tight_layout()
plt.savefig('n_mels_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Test accuracy bar chart
test_accs = [n_mels_results[m]['test_acc'] for m in N_MELS_LIST]
fig2, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar([str(m) for m in N_MELS_LIST], test_accs, color='steelblue', width=0.5)
ax.set_title('Test Accuracy vs n_mels')
ax.set_xlabel('n_mels'); ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1)
for bar, v in zip(bars, test_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig('n_mels_test_acc.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 6: Experiment — varying `groups` in Conv1d

Fix `n_mels=80`, vary `groups` ∈ {1, 2, 4, 8, 16}.  
Note: `groups` must divide both `in_channels` and `out_channels`.  
With n_mels=80 and channels 64/128, all values {1,2,4,8,16} satisfy this.

In [ ]:
GROUPS_LIST = [1, 2, 4, 8, 16]
groups_results = {}

for groups in GROUPS_LIST:
    print(f'\n=== groups={groups} ===')
    tmp_model = SpeechCNN(n_mels=80, groups=groups)
    backbone_params = tmp_model.count_backbone_parameters()

    # FLOPs for the backbone
    try:
        from thop import profile as thop_profile
        dummy_mel = torch.zeros(1, 80, 101)
        flops, _ = thop_profile(tmp_model.backbone, inputs=(dummy_mel,), verbose=False)
    except Exception:
        flops = None

    _, history, test_acc = run_experiment(n_mels=80, groups=groups, epochs=EPOCHS)
    groups_results[groups] = {
        'history': history,
        'test_acc': test_acc,
        'backbone_params': backbone_params,
        'flops': flops,
    }

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Effect of groups parameter (n_mels=80)', fontsize=14)

for g, result in groups_results.items():
    h = result['history']
    axes[0].plot(h['train_loss'],  label=f'groups={g}')
    axes[1].plot(h['val_acc'],     label=f'groups={g}')
    axes[2].plot(h['epoch_time'],  label=f'groups={g}')

for ax, title, ylabel in zip(axes,
    ['Train Loss', 'Val Accuracy', 'Epoch Time (s)'],
    ['Loss', 'Accuracy', 'Seconds']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.legend()

plt.tight_layout()
plt.savefig('groups_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary: params, FLOPs, epoch time, test accuracy vs groups
params_list     = [groups_results[g]['backbone_params'] for g in GROUPS_LIST]
flops_list      = [groups_results[g]['flops']           for g in GROUPS_LIST]
epoch_time_list = [np.mean(groups_results[g]['history']['epoch_time']) for g in GROUPS_LIST]
test_acc_list   = [groups_results[g]['test_acc']        for g in GROUPS_LIST]

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
fig2.suptitle('Groups experiment summary (n_mels=80)', fontsize=14)

axes2[0, 0].plot(GROUPS_LIST, params_list, 'o-', color='navy')
axes2[0, 0].set_title('Backbone Parameters vs groups')
axes2[0, 0].set_xlabel('groups'); axes2[0, 0].set_ylabel('Parameters')
axes2[0, 0].set_xscale('log', base=2)

if all(f is not None for f in flops_list):
    axes2[0, 1].plot(GROUPS_LIST, [f / 1e6 for f in flops_list], 'o-', color='darkred')
    axes2[0, 1].set_title('Backbone FLOPs (M) vs groups')
    axes2[0, 1].set_xlabel('groups'); axes2[0, 1].set_ylabel('FLOPs (M)')
    axes2[0, 1].set_xscale('log', base=2)
else:
    axes2[0, 1].text(0.5, 0.5, 'FLOPs unavailable', transform=axes2[0, 1].transAxes, ha='center')

axes2[1, 0].plot(GROUPS_LIST, epoch_time_list, 'o-', color='green')
axes2[1, 0].set_title('Avg Epoch Time vs groups')
axes2[1, 0].set_xlabel('groups'); axes2[1, 0].set_ylabel('Time (s)')
axes2[1, 0].set_xscale('log', base=2)

axes2[1, 1].plot(GROUPS_LIST, test_acc_list, 'o-', color='darkorange')
axes2[1, 1].set_title('Test Accuracy vs groups')
axes2[1, 1].set_xlabel('groups'); axes2[1, 1].set_ylabel('Accuracy')
axes2[1, 1].set_xscale('log', base=2)

plt.tight_layout()
plt.savefig('groups_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print(f'{'groups':>8} | {'params':>10} | {'FLOPs (M)':>10} | {'epoch_time':>10} | {'test_acc':>10}')
print('-' * 60)
for g, p, f, t, a in zip(GROUPS_LIST, params_list, flops_list, epoch_time_list, test_acc_list):
    f_str = f'{f/1e6:.2f}' if f is not None else 'N/A'
    print(f'{g:>8} | {p:>10,} | {f_str:>10} | {t:>10.2f} | {a:>10.4f}')

## Conclusions

### Part 1 — LogMelFilterBanks
*(fill in after running)*

### Part 5 — Effect of n_mels
*(fill in after running, e.g.: larger n_mels → richer features → higher accuracy but slower inference)*

### Part 6 — Effect of groups
*(fill in after running, e.g.: higher groups → fewer parameters and FLOPs → faster training, but may hurt accuracy)*